# 02 · Train ACT from scratch (Step 1c)

<a href="https://colab.research.google.com/github/danielamrh/act-visual-robustness/blob/main/notebooks/02_train_act.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Train our own ACT policy on `lerobot/aloha_sim_transfer_cube_human` (50 human demos) and compare it with the
pretrained checkpoint from notebook 01 (83 %). This run is also **the baseline all later encoder experiments
are compared against**.

Colab Free sessions die after a few hours, so training is **resumable**:
we train on the fast local disk and a background thread mirrors the newest checkpoints to Drive.
If the session dies, just reopen this notebook and run it top to bottom again: it continues from the last checkpoint on Drive.

| Section | When to run |
|---|---|
| 0–2 Setup + config | every session |
| 3 Timing test | once, before the first real run |
| 4 Train | every session until finished |
| 5–6 Evaluate + loss curve | when training is done |

**Runtime → Change runtime type → T4 GPU** before running.

## 0 · Environment check

In [ ]:
import sys, subprocess
print(sys.version)
assert sys.version_info >= (3, 12), "LeRobot 0.6.x needs Python >= 3.12"
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "No GPU - switch the runtime to T4!")

## 1 · Drive + repo + install
Code comes from GitHub (always fresh via `git pull`), outputs go to Google Drive.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO = "act-visual-robustness"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/danielamrh/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
print("commit:", COMMIT)

# labmaze (pulled in by dm-control) has no wheel for Python >= 3.13 and needs Bazel
# to build. gym-aloha never uses it, so install an empty placeholder first.
if sys.version_info >= (3, 13):
    !pip install -q ./tools/labmaze_stub

# installs lerobot[aloha] pinned in pyproject.toml (takes a few minutes)
!pip install -q -e ".[sim]"
sys.path.insert(0, f"{REPO_DIR}/src")  # editable install is only picked up after a restart

# GPU rendering: without NVIDIA's EGL registration MuJoCo silently renders on the CPU
from avr.colab import ensure_nvidia_egl, gl_renderer
ensure_nvidia_egl()
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
renderer = gl_renderer()
print("OpenGL renderer:", renderer)
if "llvmpipe" in renderer.lower() or renderer.startswith("failed"):
    print("⚠ MuJoCo is NOT rendering on the GPU - rollouts will be extremely slow")

DRIVE_ROOT = "/content/drive/MyDrive/act_robustness"
os.makedirs(DRIVE_ROOT, exist_ok=True)

### Quick check: can we create and render the ALOHA env?

In [ ]:
import gymnasium as gym
import gym_aloha  # registers gym_aloha/* envs
import matplotlib.pyplot as plt

import time
env = gym.make("gym_aloha/AlohaTransferCube-v0")
obs, _ = env.reset(seed=0)
img = obs["top"] if isinstance(obs, dict) and "top" in obs else env.render()

# speed check: env step incl. observation rendering, zero actions
t0 = time.perf_counter()
for _ in range(50):
    env.step(env.action_space.sample() * 0)
dt = (time.perf_counter() - t0) / 50
env.close()
print(f"{dt*1000:.0f} ms per env step -> one 400-step episode ≈ {400*dt:.0f} s (+ video rendering)")
plt.imshow(img); plt.axis("off"); plt.title("AlohaTransferCube-v0, seed 0");

## 2 · Run configuration
Defaults follow the LeRobot ACT recipe for ALOHA sim (the pretrained checkpoint was trained for 80k steps).
Change `RUN_NAME` for every new experiment: it is the folder name on Drive.

In [ ]:
import json, time
from avr.colab import run_streaming
from avr.background import start_background, watch
from avr.eval.chunked import run_chunked_eval
from avr.lerobot_cli import train_cmd, resume_cmd
from avr.checkpoints import CheckpointSyncer, restore_from_drive
from avr.train_log import read_train_log, seconds_per_step

RUN_NAME   = "act_transfer_cube_human_resnet18_s1000"
DATASET    = "lerobot/aloha_sim_transfer_cube_human"
TASK       = "AlohaTransferCube-v0"
STEPS      = 80_000
BATCH_SIZE = 8
SAVE_FREQ  = 5_000    # adjust after the timing test: aim for a checkpoint every ~20-30 min
SEED       = 1000
USE_AMP    = False    # fp16 would be faster on a T4 but deviates from the reference recipe

LOCAL_RUN = f"/content/outputs/{RUN_NAME}"
DRIVE_RUN = f"{DRIVE_ROOT}/runs/{RUN_NAME}"
os.makedirs(DRIVE_RUN, exist_ok=True)
print("local:", LOCAL_RUN, "\ndrive:", DRIVE_RUN)

## 3 · Timing test (run once)
500 training steps without checkpoints or rollouts. Measures seconds per step on this GPU and whether the GPU
(`updt_s`) or video decoding / data loading (`data_s`) is the bottleneck. The first call also downloads the dataset.

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
timing_dir = f"/content/outputs/timing_{stamp}"
timing_log = f"{DRIVE_ROOT}/runs/_timing/timing_{COMMIT}_{stamp}.log"

run_streaming(
    train_cmd(timing_dir, dataset=DATASET, task=TASK, steps=500, batch_size=BATCH_SIZE,
              log_freq=50, env_eval_freq=0, save_checkpoint=False, seed=SEED, use_amp=USE_AMP),
    log_path=timing_log,
)

recs = read_train_log(timing_log)
sps = seconds_per_step(recs)
updt = sum(r["updt_s"] for r in recs[1:]) / len(recs[1:])
data = sum(r["data_s"] for r in recs[1:]) / len(recs[1:])
print(f"\n{sps:.3f} s/step  (max per log window: update {updt:.3f} s, data {data:.3f} s)")
print(f"{STEPS} steps ≈ {STEPS * sps / 3600:.1f} h of training (+ in-training rollouts)")
print(f"checkpoint every ~25 min ≈ SAVE_FREQ = {max(1000, round(25 * 60 / sps, -3)):.0f}")
if data > 0.5 * updt:
    print("⚠ data loading is a large share -> video decoding on 2 CPU cores is limiting")

## 4 · Train (resumable)
Starts a fresh run, or resumes from the newest checkpoint on Drive if one exists.
Runs as a background job: the cell shows a status line every 2 min, the full log goes to `train.log` on Drive. Evaluation rollouts (10 episodes) run every 20k steps.

In [ ]:
config_path = restore_from_drive(DRIVE_RUN, LOCAL_RUN)
if config_path is not None:
    print("resuming from", os.path.realpath(config_path.parent.parent))
    cmd = resume_cmd(str(config_path))
else:
    assert not os.path.exists(LOCAL_RUN), f"{LOCAL_RUN} exists without checkpoints; delete it or change RUN_NAME"
    cmd = train_cmd(LOCAL_RUN, dataset=DATASET, task=TASK, steps=STEPS, batch_size=BATCH_SIZE,
                    save_freq=SAVE_FREQ, seed=SEED, use_amp=USE_AMP)
    meta = dict(run=RUN_NAME, commit=COMMIT, started=time.strftime("%Y-%m-%d %H:%M:%S"),
                dataset=DATASET, task=TASK, steps=STEPS, batch_size=BATCH_SIZE,
                save_freq=SAVE_FREQ, seed=SEED, use_amp=USE_AMP, cmd=cmd)
    with open(f"{DRIVE_RUN}/run_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

# background job: only a status line every 2 min in this cell, full log in train.log on Drive
train_log = f"{DRIVE_RUN}/train.log"
with CheckpointSyncer(LOCAL_RUN, DRIVE_RUN, keep=2, interval=60):
    watch(start_background(cmd, train_log), train_log, interval=120)
print("training finished")

## 5 · Evaluate the final checkpoint
Same protocol as notebook 01 so the numbers are directly comparable.

In [ ]:
import glob
from avr.eval.stats import merge_eval_infos, summarize_eval, format_summary

if not os.path.exists(f"{LOCAL_RUN}/checkpoints/last"):
    restore_from_drive(DRIVE_RUN, LOCAL_RUN)
policy_dir = os.path.realpath(f"{LOCAL_RUN}/checkpoints/last/pretrained_model")
step = os.path.basename(os.path.dirname(policy_dir))
print("evaluating step", step)

N_EPISODES = 500  # same protocol as notebook 01: chunks of 100, seeds 1000-1499, resumable
eval_root = f"{DRIVE_RUN}/eval/step{step}_n{N_EPISODES}_seed1000"
info_paths = run_chunked_eval(policy_dir, eval_root, n_episodes=N_EPISODES, chunk_size=100, seed=1000, task=TASK)

summary = summarize_eval(merge_eval_infos(info_paths))
print(format_summary(summary))
with open(f"{eval_root}/summary.json", "w") as f:
    json.dump({"commit": COMMIT, "run": RUN_NAME, "step": int(step), **summary}, f, indent=2)

## 6 · Training curve

In [ ]:
recs = read_train_log(f"{DRIVE_RUN}/train.log")
steps = [r["step"] for r in recs]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(steps, [r["loss"] for r in recs])
ax.set_yscale("log"); ax.set_xlabel("step (approx., LeRobot rounds to 1K)"); ax.set_ylabel("train loss (L1 + KL)")
ax.set_title(RUN_NAME); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{DRIVE_RUN}/loss_curve.png", dpi=150)